# Deep Research Tool — 全機能ガイド

`deep_research_tool` の全機能を設定項目ごとに解説するリファレンスノートブックです。
各セルは独立して実行できます（APIキー等は自分の値に置き換えてください）。

## 目次
1. インストールとセットアップ
2. LLMプロバイダ設定（APIキー・カスタムエンドポイント）
3. 工程別LLM切り替え（stage_llm）
4. Web検索設定（DuckDuckGo / Selenium・WebDriverパス）
5. クロールモード（標準 / 高速 / AIクロール）
6. 情報源モード（web / local / hybrid）
7. 多言語検索
8. DeepThink（推論強化）
9. フェルミ推定
10. レポート生成（V1 / V2 / V3・文体・分量・図表）
11. 検証・エビデンス・警告
12. 実行と結果の取得


## 1. インストールとセットアップ

```bash
# リポジトリ直下で
pip install -e .

# Selenium検索 / AIクロール（ブラウザ）を使う場合
pip install -e ".[selenium]"

# 全機能（PDF出力・pint単位換算など含む）
pip install -e ".[all]"
```

APIキーは環境変数でも渡せます:

| 環境変数 | 用途 |
|---|---|
| `OPENAI_API_KEY` | OpenAI APIキー |
| `ANTHROPIC_API_KEY` | Anthropic APIキー |
| `OPENAI_BASE_URL` | OpenAIのカスタムエンドポイント |
| `ANTHROPIC_BASE_URL` | Anthropicのカスタムエンドポイント |
| `LOCAL_LLM_BASE_URL` | ローカルLLMサーバーURL |
| `HTTPS_PROXY` / `HTTP_PROXY` | プロキシ |
| `SELENIUM_DRIVER_PATH` | WebDriver実行ファイルのパス |


In [ ]:
# 動作確認: パッケージが読み込めるか
import deep_research_tool
from deep_research_tool.config import create_config
from deep_research_tool.main import DeepResearchTool
print("deep_research_tool loaded")

## 2. LLMプロバイダ設定

`provider` は `openai` / `anthropic` / `local` の3種。

**カスタムエンドポイント（base_url）**: 社内APIゲートウェイ、Azure系デプロイ、
OpenAI互換サーバー（vLLM等）を経由する場合は `openai_base_url` /
`anthropic_base_url` を指定します。未指定なら公式エンドポイントです。


In [ ]:
from deep_research_tool.config import create_config

# --- OpenAI（公式エンドポイント） ---
config = create_config(
    provider="openai",
    openai_api_key="sk-...",          # 省略時は環境変数 OPENAI_API_KEY
    model="gpt-5-mini",
)

# --- OpenAI互換の社内ゲートウェイ経由 ---
config_gw = create_config(
    provider="openai",
    openai_api_key="sk-...",
    openai_base_url="https://gateway.example.com/v1",  # ★カスタムエンドポイント
    model="gpt-5-mini",
)

# --- Anthropic ---
config_claude = create_config(
    provider="anthropic",
    anthropic_api_key="sk-ant-...",
    anthropic_base_url=None,           # None = 公式 https://api.anthropic.com
    model="claude-3-5-sonnet-20241022",
)

# --- ローカルLLM（Ollama / vLLM / OpenAI互換） ---
config_local = create_config(
    provider="local",
    local_base_url="http://localhost:11434",  # OllamaのURL
    local_backend="ollama",                   # "ollama" / "vllm" / "openai_compatible"
    model="llama3.1:8b",
)
print("configs created")

## 3. 工程別LLM切り替え（stage_llm）

パイプラインの4工程ごとに別のLLMを割り当てられます。コストの高い執筆だけ高性能
モデルにし、機械的な評価は安価なモデルに逃がす構成が典型です。

| 工程 | 内容 |
|---|---|
| `planning` | 調査計画・目次・検索クエリ生成 |
| `crawling` | クロール時のリンク選択・関連度判断 |
| `evaluation` | 重要度採点・品質評価・一貫性チェック |
| `writing` | 本文執筆・要約・推敲 |

各工程の指定には `provider` / `model` / `api_key` / `base_url` / `backend` が使えます。


In [ ]:
config = create_config(
    provider="openai",
    openai_api_key="sk-...",
    model="gpt-5-mini",                    # 既定モデル
    stage_llm={
        # 計画と評価は軽量モデル
        "planning":   {"provider": "openai", "model": "gpt-5-nano"},
        "evaluation": {"provider": "openai", "model": "gpt-5-nano"},
        # 執筆だけ高性能モデル（プロバイダ混在も可）
        "writing": {
            "provider": "anthropic",
            "model": "claude-3-5-sonnet-20241022",
            "api_key": "sk-ant-...",
            # "base_url": "https://gateway.example.com",  # 工程単位のエンドポイントも可
        },
    },
)

## 4. Web検索設定

### DuckDuckGo（既定・軽量）
- `search_region`: 検索地域（`jp-jp` 日本 / `wt-wt` 世界）
- `safe_search`: `off` / `moderate` / `strict`

### Selenium（実ブラウザ・JS描画サイト対応）
- `browser`: `chrome` / `edge` / `firefox`
- `headless`: ヘッドレス実行（既定 True）
- `driver_path`: **WebDriver実行ファイルのパス**

**driver_path について**: 未指定の場合は webdriver-manager がドライバを自動
ダウンロードしますが、社内プロキシ・オフライン環境では失敗します。その場合は
ブラウザのバージョンに合った WebDriver（msedgedriver / chromedriver /
geckodriver）を手動配置し、そのパスを指定してください。
ダウンロードに失敗したときは Selenium 4.6+ 内蔵の Selenium Manager にも
フォールバックします。


In [ ]:
# --- DuckDuckGo（日本語結果優先） ---
config_ddg = create_config(
    provider="openai", openai_api_key="sk-...",
    search_method="duckduckgo",
    search_region="jp-jp",
    safe_search="moderate",
)

# --- Selenium + Edge + ローカルWebDriver（社内PCの典型構成） ---
config_edge = create_config(
    provider="openai", openai_api_key="sk-...",
    search_method="selenium",
    browser="edge",
    driver_path=r"C:\tools\msedgedriver.exe",  # ★ドライバの実体パス
    headless=True,
    implicit_wait=10,
)
# 環境変数 SELENIUM_DRIVER_PATH でも指定可能

## 5. クロールモード（crawl_mode）

| モード | 説明 |
|---|---|
| `standard` | 検索結果を順に取得（既定・確実） |
| `fast_batch` | 並列取得 + バッチLLM評価（高速） |
| `fast_parallel` | 並列取得 + 並列LLM評価（最速・API消費大） |
| `aicrawl` | LLMがページを読み辿るリンクを判断（requestsベース） |
| `ai_crawl_selenium` | aicrawlのSelenium版。JS描画サイトも読める |

関連パラメータ: `ai_crawl_max_total_pages`（章あたり取得上限）、
`ai_crawl_max_depth`（リンク深さ）、`ai_crawl_site_depth`（同一サイト内深さ）、
`ai_crawl_max_llm_calls`（判断コール予算）、`ai_crawl_max_pages_per_domain`、
`ai_crawl_politeness_delay`（同一ドメインへの間隔秒）。

さらに `extended_mode=True` でサイト深掘りクロール
（`crawl_max_pages` / `crawl_max_depth` / `crawl_max_sites`）も併用できます。


In [ ]:
# AIクロール（ブラウザ版）: Selenium設定（browser / driver_path）を共有します
config_crawl = create_config(
    provider="openai", openai_api_key="sk-...",
    crawl_mode="ai_crawl_selenium",
    browser="edge",
    driver_path=r"C:\tools\msedgedriver.exe",
    ai_crawl_max_total_pages=15,
    ai_crawl_site_depth=2,
    ai_crawl_max_llm_calls=25,
    ai_crawl_politeness_delay=1.0,
)

## 6. 情報源モード（source_mode）

| モード | 説明 |
|---|---|
| `web` | Web検索のみ（既定） |
| `local` | ローカル文書のみ。ネットワーク接続不要 |
| `hybrid` | ローカル文書 + Web検索の併用 |

対応形式: PDF / DOCX / XLSX / PPTX / TXT / MD / CSV
文書は `run()` の `additional_documents` に渡します。


In [ ]:
config_local_docs = create_config(
    provider="openai", openai_api_key="sk-...",
    source_mode="local",   # または "hybrid"
)
# 実行時:
# tool = DeepResearchTool(config_local_docs)
# result = tool.run(
#     query="社内資料に基づく○○の現状分析",
#     additional_documents=["./docs/report2025.pdf", "./docs/data.xlsx"],
# )

## 7. 多言語検索

複数言語で同時に検索し、結果を統合します。


In [ ]:
config_ml = create_config(
    provider="openai", openai_api_key="sk-...",
    multilingual=True,
    search_languages=["ja", "en", "zh"],  # 日英中で検索
    results_per_language=5,
    query_translation="llm",   # クエリをLLMで各言語に翻訳
    translate_results=True,    # 結果を出力言語へ翻訳
)

## 8. DeepThink（推論強化）

収集した情報に対して多段の推論・一貫性チェック・原典忠実度チェックをかけます。

- `deep_think_level`: 0.0（保守的）〜 1.0（探索的）
- `consistency_mode`: `warn`（警告のみ）/ `revise`（自動修正）/ `strict`（厳格）


In [ ]:
config_dt = create_config(
    provider="openai", openai_api_key="sk-...",
    deep_think=True,
    deep_think_level=0.5,
    reasoning_iterations=2,
    consistency_threshold=0.7,
    consistency_mode="revise",
    fidelity_threshold=0.6,
)

## 9. フェルミ推定

統計が存在しない定量的な問いを分解木で推定します。エビデンス自動照合・感度分析・
モンテカルロシミュレーション・低信頼度リーフの再帰的サブ分解に対応。


In [ ]:
config_fermi = create_config(
    provider="openai", openai_api_key="sk-...",
    fermi_estimation=True,
    fermi_auto_detect=True,            # 問いから推定対象を自動検出
    fermi_target_metrics=None,         # 明示指定も可: ["国内の年間○○台数"]
    fermi_max_tree_depth=4,
    fermi_max_leaf_nodes=10,
    fermi_monte_carlo=1000,            # シミュレーション回数
    fermi_include_sensitivity=True,    # 感度分析
    fermi_enable_sub_decomposition=True,   # 低信頼度リーフの再分解
    fermi_sub_decomposition_confidence_threshold=0.65,
)

## 10. レポート生成

### 生成エンジン（report_generator_version）
| バージョン | 説明 |
|---|---|
| `v1` | 章ごとに執筆。多形式出力（markdown / docx / pdf） |
| `v2` | 用語統一・章間コンテキスト引き継ぎ・一貫性チェック・自然さ推敲 |
| `v3` | python-docx APIでDOCXを直接構築。図表・引用の配置精度が最も高い |

### 分量・文体
- `target_pages` / `target_characters`: ページ数・文字数目標（章ごとに配分）
- `v2_writing_style`: `formal`（である調）/ `business`（です・ます調）/ `technical` / `executive` / `casual`
- `v2_target_audience`: `expert` / `business` / `engineer` / `general` / `student`
- `v2_technical_level`: 1〜5

### 図表
- `auto_figures=True`: 図表を自動生成してレポートに挿入
- `chart_library`: `matplotlib` / `seaborn`
- `intelligent_charts` / `chart_insights` / `chart_max_per_section`
- `numerical_extraction`: エビデンスからの数値抽出、`derived_metrics`: CAGR等の派生指標


In [ ]:
config_report = create_config(
    provider="openai", openai_api_key="sk-...",
    # 生成エンジンと形式
    report_generator_version="v2",
    output_format="docx",
    output_dir="./output",
    # 分量
    target_pages=15,             # または target_characters=30000
    # 文体（V2）
    v2_writing_style="business",
    v2_target_audience="business",
    v2_technical_level=3,
    v2_enable_consistency_check=True,
    v2_enable_two_phase=True,    # 草稿→推敲の2段階生成
    v2_enable_polish=True,       # 日本語の自然さ最終推敲
    v2_include_glossary=True,    # 用語集を付ける
    # 図表
    auto_figures=True,
    chart_library="seaborn",
    auto_figures_max_images=2,
    intelligent_charts=True,
    chart_insights=True,
    chart_max_per_section=3,
    numerical_extraction=True,
    derived_metrics=True,
)

## 11. 検証・エビデンス・警告

- `enable_verification=True`: ハルシネーション検証。原典と突き合わせ、HTMLレポートを出力
- `save_evidence` / `evidence_format`: 収集エビデンスを `json` / `csv` / `both` で保存
- `content_filter_mode`: `strict` / `moderate` / `minimal` / `none`（広告・スパム除去）
- `custom_blocked_domains` / `custom_whitelisted_domains`
- **ResearchWarnings**: 実行中のフォールバック（抽出失敗等）を重要度別に収集し、
  レポート末尾に「処理中の警告・注意事項」として自動表示


In [ ]:
config_verify = create_config(
    provider="openai", openai_api_key="sk-...",
    enable_verification=True,
    save_evidence=True,
    evidence_format="both",
    content_filter_mode="moderate",
    custom_blocked_domains=["example-spam.com"],
    custom_whitelisted_domains=["stat.go.jp"],
    verbose=True,
    log_file="./output/research.log",
)

## 12. 実行と結果の取得

`DeepResearchTool(config).run(...)` が調査からレポート生成までを一括実行します。


In [ ]:
from deep_research_tool.main import DeepResearchTool

config = create_config(
    provider="openai",
    openai_api_key="sk-...",
    search_region="jp-jp",
    research_iterations=3,
    max_pages_per_query=3,
    output_format="markdown",
    output_dir="./output",
    language="ja",
)

tool = DeepResearchTool(config)

def on_progress(message, percentage):
    print(f"[{percentage:5.1f}%] {message}")

result = tool.run(
    query="国内製造業における生成AI活用の現状と課題",
    requirements="導入事例・投資動向・規制面を含めること",
    additional_context="",          # 事前知識があれば渡す
    additional_documents=None,      # ローカル文書のリスト
    progress_callback=on_progress,
)

print("レポート:", result["report_path"])
print("エビデンスJSON:", result["evidence_json"])
print("エビデンスCSV:", result.get("evidence_csv"))
print("検証レポート:", result.get("verification_html"))